In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import requests,json,hashlib,tempfile,shutil
from datetime import datetime,timezone
ROOT=Path('/content/drive/MyDrive/Allinone/Exercise_Quality_95_Target_20261028');ROOT.mkdir(parents=True,exist_ok=True)
url='https://softwarecenter.qualcomm.com/api/download/software/dataset/AIDataset/Qualcomm_Exercise_Video_Dataset/QEVD-FIT-300k-Additional-Labels/fine_grained_labels_with_worker_ids.json'
report={'source':'QEVD official participant labels','source_url':url,'utc':datetime.now(timezone.utc).isoformat(),'raw_data_saved_to_drive':False,'status':'pending','not_used_for_training_yet':True}
try:
 r=requests.get(url,timeout=(20,120));report.update(http_status=r.status_code,content_type=r.headers.get('content-type'),resolved_url=r.url)
 r.raise_for_status()
 data=r.json();report['json_type']=type(data).__name__;report['keys']=list(data)[:20] if isinstance(data,dict) else None
 rows=data if isinstance(data,list) else next((v for v in data.values() if isinstance(v,list)),[])
 report['rows']=len(rows);report['example_rows']=rows[:3]
 labels={}
 for row in rows:
  if not isinstance(row,dict):continue
  for label in row.get('labels',[]):labels[label]=labels.get(label,0)+1
 report['label_counts']=labels
 report['status']='metadata_accessible' if rows else 'schema_needs_inspection'
 # Save only relevant annotation metadata; do not infer GOOD/BAD from exercise identity.
 terms=['push','squat','bicep','curl','shoulder','overhead','row','tricep','side bend']
 selected=[x for x in rows if isinstance(x,dict) and any(t in json.dumps(x.get('labels',[])).lower() for t in terms)]
 if selected:
  import gzip
  with gzip.open(ROOT/'qevd_relevant_annotation_metadata.json.gz','wt',encoding='utf-8') as f:json.dump(selected,f)
  report['selected_rows']=len(selected)
except Exception as e:report.update(status='access_or_schema_failed',error=repr(e))
(ROOT/'data_access_probe.json').write_text(json.dumps(report,ensure_ascii=False,indent=2));print(json.dumps({k:v for k,v in report.items() if k!='label_counts'},ensure_ascii=False,indent=2));print('PROBE_COMPLETE')
